Cell 1 (code): environment check. The assert catches a version mismatch with Colab.

In [1]:
import sys, platform, importlib.metadata as md
print(sys.version)
print(platform.platform())
ver = md.version("pycolmap")
print("pycolmap", ver)
assert ver == "4.2.1", f"Colab is pinned to 4.2.1, local is {ver}"

3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Windows-11-10.0.26300-SP0
pycolmap 4.2.1


Cell 2 (code, optional): install. Run it only if cell 1 fails, then restart the kernel.

In [2]:
# %pip install pycolmap==4.2.1

Cell 3 (code): print the real API. Same as Colab cell 3, so you can diff the two outputs.

In [3]:
import pycolmap
for fn in ["extract_features", "match_sequential", "incremental_mapping", "undistort_images"]:
    print("=" * 20, fn)
    print(getattr(pycolmap, fn).__doc__)
print("ImageReaderOptions:", [m for m in dir(pycolmap.ImageReaderOptions) if not m.startswith("_")])
print("CameraMode:", list(pycolmap.CameraMode.__members__))
print("SequentialPairingOptions:", [m for m in dir(pycolmap.SequentialPairingOptions) if not m.startswith("_")])


==================== extract_features
extract_features(database_path: os.PathLike | str | bytes, image_path: os.PathLike | str | bytes, image_names: collections.abc.Sequence[str] = [], camera_mode: pycolmap._core.CameraMode = CameraMode.AUTO, reader_options: pycolmap._core.ImageReaderOptions = ImageReaderOptions(), extraction_options: pycolmap._core.FeatureExtractionOptions = FeatureExtractionOptions(), device: pycolmap._core.Device = Device.auto, cancellation_token: pycolmap._core.CancellationToken = None) -> None

Extract SIFT Features and write them to database

==================== match_sequential
match_sequential(database_path: os.PathLike | str | bytes, matching_options: pycolmap._core.FeatureMatchingOptions = FeatureMatchingOptions(), pairing_options: pycolmap._core.SequentialPairingOptions = SequentialPairingOptions(), verification_options: pycolmap._core.TwoViewGeometryOptions = TwoViewGeometryOptions(), device: pycolmap._core.Device = Device.auto, cancellation_token: pycolma

Cell 4 (code): config. Edit VIDEO_NAME. Keep the other three values identical to the Colab run.

In [4]:
from pathlib import Path
import importlib
import sfm_pipeline                      # same folder as this notebook
importlib.reload(sfm_pipeline)           # re-run after editing the module

ROOT = Path.cwd().parents[2]             # .../Case_Study
assert (ROOT / "source").exists(), f"Unexpected working directory: {Path.cwd()}"

# current videos:
# IMG_9175
# IMG_9176
VIDEO_NAME = "IMG_9176"              # <- folder under source/frames/
FRAMES_SRC = ROOT / "source" / "frames" / VIDEO_NAME

N_FRAMES     = 40                        # match the Colab run; None = all frames
CAMERA_MODEL = "OPENCV"
OVERLAP      = 20

RUN_NAME = f"run_{N_FRAMES}frames_local" if N_FRAMES else "run_full_local"
OUT = ROOT / "data" / "colmap" / VIDEO_NAME / RUN_NAME
print("frames:", FRAMES_SRC, "| exists:", FRAMES_SRC.exists())
print("output:", OUT)


frames: c:\Users\Andy\Desktop\school\Regis Stuff\Classes\2026\fall_2\MSDS_688_AI\Case_Study\source\frames\IMG_9176 | exists: True
output: c:\Users\Andy\Desktop\school\Regis Stuff\Classes\2026\fall_2\MSDS_688_AI\Case_Study\data\colmap\IMG_9176\run_40frames_local


Cell 5 (code): run the pipeline. run_sfm deletes OUT first, so a rerun starts clean.

In [5]:
rec, summary = sfm_pipeline.run_sfm(
    FRAMES_SRC, OUT,
    n_frames=N_FRAMES, camera_model=CAMERA_MODEL, overlap=OVERLAP,
)


Using 40 frames
models found:            1  (registered per model: {0: 40})
best model id:           0
registered / total:      40 / 40
sparse 3D points:        24779
observations:            121251
mean reprojection error: 0.3991 px
unregistered frames (0): []


Cell 6 (code): inspect the output layout. This is the same check as Colab cell 7.

In [6]:
for p in sorted(OUT.rglob("*")):
    if p.suffix.lower() != ".jpg":
        print(p.relative_to(OUT))


database.db
images
sparse
sparse\0
sparse\0\cameras.bin
sparse\0\frames.bin
sparse\0\images.bin
sparse\0\points3D.bin
sparse\0\rigs.bin
sparse_best
sparse_best\cameras.bin
sparse_best\frames.bin
sparse_best\images.bin
sparse_best\points3D.bin
sparse_best\rigs.bin
sparse_points.ply
summary.txt
undistorted
undistorted\images
undistorted\run-colmap-geometric.sh
undistorted\run-colmap-photometric.sh
undistorted\sparse
undistorted\sparse\cameras.bin
undistorted\sparse\frames.bin
undistorted\sparse\images.bin
undistorted\sparse\points3D.bin
undistorted\sparse\rigs.bin
undistorted\stereo
undistorted\stereo\consistency_graphs
undistorted\stereo\depth_maps
undistorted\stereo\fusion.cfg
undistorted\stereo\normal_maps
undistorted\stereo\patch-match.cfg


Full Run test

In [ ]:
# import time
# #IMG_9175
# #IMG_9176
# for v in ["IMG_9175"]:#, "IMG_9176"]:
#     out = ROOT / "data" / "colmap" / v / "run_full_local"
#     t0 = time.time()
#     rec, summary = sfm_pipeline.run_sfm(
#         ROOT / "source" / "frames" / v, out,
#         n_frames=None, camera_model=CAMERA_MODEL, overlap=OVERLAP,
#     )
#     print(f"{v}: {(time.time()-t0)/60:.1f} min\n")

visualization

In [ ]:
# import numpy as np, matplotlib.pyplot as plt, pycolmap

# def cam_center(im):
#     try:
#         return np.asarray(im.projection_center())
#     except AttributeError:
#         T = im.cam_from_world() if callable(im.cam_from_world) else im.cam_from_world
#         M = T.matrix()
#         return -M[:, :3].T @ M[:, 3]

# def check_path(run_dir):
#     rec = pycolmap.Reconstruction(str(run_dir / "sparse_best"))
#     imgs = sorted(rec.images.values(), key=lambda im: im.name)
#     C = np.array([cam_center(im) for im in imgs])
#     P = np.array([p.xyz for p in rec.points3D.values()])
#     step = np.linalg.norm(np.diff(C, axis=0), axis=1)
#     print(f"cameras {len(C)} | median step {np.median(step):.4g} | "
#           f"max step {step.max():.4g} | max/median {step.max()/np.median(step):.1f}")
#     print("largest steps after frame index:", np.argsort(step)[-5:][::-1])

#     lo, hi = np.percentile(P, 1, axis=0), np.percentile(P, 99, axis=0)
#     Pc = P[np.all((P >= lo) & (P <= hi), axis=1)]

#     fig = plt.figure(figsize=(14, 12))
#     for i, (el, az, t) in enumerate([(20, 45, "Perspective"), (90, -90, "Top-down"),
#                                      (0, 0, "Side X"), (0, 90, "Side Y")], 1):
#         ax = fig.add_subplot(2, 2, i, projection="3d")
#         ax.scatter(*Pc[::5].T, s=0.3, c="gray", alpha=0.5)
#         ax.plot(*C.T, "r.-", ms=2, lw=1)
#         ax.scatter(*C[0], c="green", s=40, label="start")
#         ax.scatter(*C[-1], c="blue", s=40, label="end")
#         ax.set_box_aspect(np.ptp(np.vstack([Pc, C]), axis=0))
#         ax.view_init(elev=el, azim=az); ax.set_title(t)
#     ax.legend(); plt.tight_layout()
#     plt.savefig(run_dir / "camera_path.png", dpi=120); plt.show()

# for v in ["IMG_9175", "IMG_9176"]:
#     check_path(ROOT / "data" / "colmap" / v / "run_full_local")

focal length comparison

In [ ]:
# for v in ["IMG_9175", "IMG_9176"]:
#     r = pycolmap.Reconstruction(str(ROOT/"data"/"colmap"/v/"run_full_local"/"sparse_best"))
#     for c in r.cameras.values():
#         print(v, c)